# Lunar Orbit Mock Data and Linear Inversion

Simulate auto-correlations for two dipole antennas mounted on a torque-free
tumbling spacecraft in a 150 km equatorial lunar orbit.  Then perform a
linear inversion (Tikhonov regularisation) to jointly recover:

- per-pixel GSM brightness temperature (nside = 8)
- lunar regolith temperature  T_reg  (scalar)
- sun effective temperature  T_sun  (scalar)

The observation equation is

$$T_\text{ant}^{(i,d)} = \frac{\displaystyle\sum_j B_d(\hat n_j,t_i)\,T_\text{sky}^{(i)}(j) + T_\text{sun}\,B_d(\hat n_{\odot,i},t_i)\,m_i(j_{\odot,i})}{\displaystyle\sum_j B_d(\hat n_j,t_i)}$$

where

$$B_d(\hat n) = \left[\frac{\cos(k h_d\,\hat d\cdot\hat n) - \cos(k h_d)}{\sqrt{1-(\hat d\cdot\hat n)^2}}\right]^2$$

is the frequency-dependent center-fed dipole power pattern,
$k = 2\pi f/c$ is the wavenumber, $h_d = L_d/2$ is the half-length of dipole $d$,
$m_i(j)\in\{0,1\}$ is the lunar-occultation mask, and
$T_\text{sky}^{(i)}(j)=T_\text{GSM}(j)\,m_i(j)+T_\text{reg}\,(1-m_i(j))$.

The on-axis limit ($\hat d\cdot\hat n \to \pm 1$, $\sin\theta \to 0$) is zero for
all $kh_d$ (numerator vanishes first by L'Hôpital).

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import healpy
from scipy.spatial.transform import Rotation
from astropy.time import Time
from astropy.coordinates import get_body
import astropy.units as u
from pygdsm import GlobalSkyModel16 as GSM16

from eigsep_sim.lunar_orbit import LunarOrbit, circular_orbital_period
from eigsep_sim._observer import ICRS2GAL
from eigsep_sim.const import R_MOON

# lunar_spinner lives in the notebooks/ directory
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from lunar_spinner import make_x_points, ARM_LENGTHS, ARM_MASSES, OPENING_ANGLE_DEG, L_HAT

## Configuration

In [ ]:
# --- sky / frequency -------------------------------------------------------
FREQ        = 100e6          # Hz
NSIDE       =  8             # HEALPix resolution
NPIX        = healpy.nside2npix(NSIDE)

# --- truth values ----------------------------------------------------------
T_REGOLITH  = 300.0          # K, lunar surface temperature
T_SUN       = 5000.0         # K, effective sun pixel temperature at nside=8, 100 MHz
SIGMA_NOISE = 1.0            # K, per-observation thermal noise

# --- orbit -----------------------------------------------------------------
ALTITUDE    = 150e3          # m, above lunar surface
T_ORBITAL   = circular_orbital_period(ALTITUDE)   # s

# Equatorial orbit: orbital plane normal = celestial north pole in galactic frame.
ROT_ORBIT_VEC = ICRS2GAL @ np.array([0.0, 0.0, 1.0])   # CNP in galactic coords
START_POS     = ICRS2GAL @ np.array([1.0, 0.0, 0.0])   # vernal equinox in galactic coords

# --- observation schedule --------------------------------------------------
N_DAYS      = 60
OBS_EPOCH   = Time("2025-01-01")
N_OBS       = 4*3600
N_ROWS      = N_OBS * 2      # total rows in design matrix (2 dipoles × N_OBS times)

# --- dipole geometry (from lunar_spinner.py) -------------------------------
# ARM_LENGTHS, ARM_MASSES, OPENING_ANGLE_DEG, L_HAT imported above

# --- inversion -------------------------------------------------------------
print(f"Orbital period: {T_ORBITAL/3600:.2f} h")
print(f"Pixels: {NPIX},  Observations: {N_ROWS} (2 dipoles × {N_OBS} times)")
print(f"Arm lengths: {ARM_LENGTHS} m,  masses: {ARM_MASSES} kg,  "
      f"opening angle: {OPENING_ANGLE_DEG}°")
print(f"ROT_ORBIT_VEC (galactic): {ROT_ORBIT_VEC.round(4)}")
print(f"START_POS     (galactic): {START_POS.round(4)}")

## Spacecraft orientations

The L and I of the spinner are designed so that the torque-free precession
is ergodic — the body visits all orientations with roughly equal probability
over a month.  We therefore model the attitude at each observation epoch as
an independent draw from the uniform (Haar) measure on SO(3), which is both
the correct limiting distribution and orders of magnitude cheaper than
integrating the full rigid-body ODE.

In [ ]:
# Dipole body-frame unit vectors (from the X-arm geometry in lunar_spinner.py)
_, arm_axes = make_x_points(ARM_LENGTHS, OPENING_ANGLE_DEG)
U_BODY = np.array([u / np.linalg.norm(u) for u in arm_axes])  # (2, 3)
print(f"Dipole body-frame axes:\n  dipole 0: {U_BODY[0]}\n  dipole 1: {U_BODY[1]}")

# Sample N_OBS orientations uniformly on SO(3) — the Haar measure.
rots_obs = Rotation.random(N_OBS, random_state=42)
print(f"Sampled {N_OBS} orientations from SO(3)")

## Orbital setup, GSM, and Sun positions

In [ ]:
# LunarOrbit observer (orbital period computed automatically from altitude)
orbit = LunarOrbit(
    altitude=ALTITUDE,
    rot_orbit_vec=ROT_ORBIT_VEC,
    rot_spin_vec=[0, 0, 1],   # not used for attitude; spin handled by spinner
    start_pos=START_POS,
    spin_period=0.0,
    t0=OBS_EPOCH,
)

# Uniformly-spaced observation times (physical seconds from epoch)
t_obs_s   = np.linspace(0.0, N_DAYS * 86400.0, N_OBS, endpoint=False)
obs_times = OBS_EPOCH + t_obs_s * u.s

# GSM sky at FREQ, downsampled to NSIDE
gsm_model  = GSM16(freq_unit="Hz", data_unit="TRJ", resolution="lo", include_cmb=True)
gsm_full   = gsm_model.generate([FREQ]).astype(float)   # (npix_gsm,)
gsm_nside  = healpy.npix2nside(gsm_full.shape[0])
gsm_map    = healpy.ud_grade(gsm_full, NSIDE) if gsm_nside != NSIDE else gsm_full
print(f"GSM at {FREQ/1e6:.0f} MHz (nside={NSIDE}): "
      f"min={gsm_map.min():.0f} K, mean={gsm_map.mean():.0f} K, max={gsm_map.max():.0f} K")

# Galactic pixel unit vectors  (3, npix)
N_GAL = np.array(healpy.pix2vec(NSIDE, np.arange(NPIX)))  # (3, npix)

# Batch-query Sun positions in galactic frame
# (Earth–Moon parallax ≪ pixel size at nside=8, so Earth centre is fine)
print("Querying Sun positions … ", end="", flush=True)
sun_coords = get_body("sun", obs_times)          # SkyCoord array
sun_gal    = sun_coords.galactic
l_s = sun_gal.l.rad
b_s = sun_gal.b.rad
# Sun unit vectors in galactic frame, (3, N_OBS)
N_SUN = np.array([
    np.cos(b_s) * np.cos(l_s),
    np.cos(b_s) * np.sin(l_s),
    np.sin(b_s),
])
J_SUN = healpy.vec2pix(NSIDE, *N_SUN)           # (N_OBS,) sun pixel indices
print("done")

## Precompute per-observation quantities

For each observation time we need:
- lunar occultation mask  $m_i(j)$  from `LunarOrbit.above_horizon`
- dipole axes in galactic frame from the spinner quaternion
- dipole beam patterns  $B_0, B_1$  over all sky pixels

In [ ]:
# Wavenumber and per-dipole half-lengths  (ARM_LENGTHS are full tip-to-tip)
C_LIGHT = 2.998e8                                    # m/s
KH = np.pi * np.array(ARM_LENGTHS) * FREQ / C_LIGHT  # kh_d = π L_d f / c  (2,)
#print(f"Dipole electrical half-lengths  kh = {KH.round(4)} rad  "
#      f"({KH/np.pi:.3f} × π;  λ/2 would be π/2 = {np.pi/2:.3f})")

# Pre-allocate caches
masks  = np.empty((N_OBS, NPIX), dtype=np.float32)  # 1 = open, 0 = blocked
beams  = np.empty((N_OBS, 2, NPIX), dtype=np.float32)  # beam for each dipole

print("Precomputing masks and beams … ", end="", flush=True)
for i, t in enumerate(obs_times):
    # Lunar occultation mask
    orbit.set_time(t)
    masks[i] = orbit.above_horizon(NSIDE).astype(np.float32)

    # Dipole axes in galactic frame  (2, 3)
    D_gal = rots_obs[i].apply(U_BODY)   # (2, 3)

    # Center-fed dipole beam: B(θ) = [(cos(kh·cosθ) − cos(kh)) / sinθ]²
    # cos θ = d̂·n̂;  sin²θ = 1 − cos²θ;  on-axis limit = 0 for all kh
    cos_t = D_gal @ N_GAL                                # (2, npix)
    sin2  = np.maximum(1.0 - cos_t ** 2, 0.0)           # (2, npix)
    kh    = KH[:, np.newaxis]                            # (2, 1) for broadcast
    numer = np.cos(kh * cos_t) - np.cos(kh)             # (2, npix)
    beams[i] = np.where(sin2 > 0,
                        numer ** 2 / sin2,
                        0.0).astype(np.float32)

print("done")

# Beam solid angles: Σ_j B(n_j) over all pixels
omega_B = beams.sum(axis=2)   # (N_OBS, 2)
print(f"Mean beam solid angles (pixels): dipole 0 = {omega_B[:, 0].mean():.1f},  "
      f"dipole 1 = {omega_B[:, 1].mean():.1f}")

## Forward model — generate mock observations

$$T_\text{ant}^{(i,d)} = \frac{\sum_j B_d^{(i)}(j)\,T_\text{sky}^{(i)}(j) + T_\text{sun}\,B_d^{(i)}(j_{\odot,i})\,m_i(j_{\odot,i})}{\sum_j B_d^{(i)}(j)}$$

In [ ]:
# Sky temperature in galactic frame (npix,)
# masks selects open sky (GSM) vs Moon-blocked (regolith)

rng  = np.random.default_rng(42)
data = np.empty((N_OBS, 2))   # T_ant for (time, dipole)

for i in range(N_OBS):
    m = masks[i]                                # (npix,)  1=open, 0=blocked

    # Sky including lunar occultation
    T_sky = gsm_map * m + T_REGOLITH * (1.0 - m)   # (npix,)

    # Sun: point-source contribution (only if not eclipsed by Moon)
    sun_mask_i = m[J_SUN[i]]                    # 0 if sun is behind Moon

    for d in range(2):
        B   = beams[i, d]                        # (npix,)
        OmB = omega_B[i, d]

        T_ant = (
            np.dot(B, T_sky)
            + T_SUN * B[J_SUN[i]] * sun_mask_i
        ) / OmB

        data[i, d] = T_ant + rng.normal(scale=SIGMA_NOISE)

y = data.ravel()   # (N_OBS * 2,)  — the measurement vector
print(f"T_ant range: {y.min():.1f} – {y.max():.1f} K  "
      f"(mean {y.mean():.1f} K)")

## Design matrix — per-pixel parameterisation

We solve directly for the temperature of each HEALPix pixel plus two scalars.
The observation row for dipole $d$ at time $i$ is

$$A[r,\,j] = \frac{B_d^{(i)}(j)\,m_i(j)}{\Omega_B^{(i,d)}}, \qquad
  A[r,\,N_\text{pix}] = \frac{\sum_j B_d^{(i)}(j)(1-m_i(j))}{\Omega_B^{(i,d)}}, \qquad
  A[r,\,N_\text{pix}+1] = \frac{B_d^{(i)}(j_{\odot,i})\,m_i(j_{\odot,i})}{\Omega_B^{(i,d)}}$$

The forward model is exact: the only residual in $\mathbf{A}\mathbf{x}_\text{true}-\mathbf{y}$ is thermal noise.

In [ ]:
N_UNKNOWNS = NPIX + 2   # per-pixel GSM + T_reg + T_sun

# Build design matrix  A : (N_ROWS, N_UNKNOWNS)
A = np.zeros((N_ROWS, N_UNKNOWNS), dtype=np.float64)

for i in range(N_OBS):
    m_i        = masks[i].astype(np.float64)
    sun_mask_i = m_i[J_SUN[i]]

    for d in range(2):
        r   = 2 * i + d
        B   = beams[i, d].astype(np.float64)
        OmB = omega_B[i, d]

        A[r, :NPIX]   = B * m_i / OmB                          # open-sky pixels
        A[r,  NPIX]   = np.dot(B, 1.0 - m_i) / OmB            # T_reg
        A[r,  NPIX+1] = B[J_SUN[i]] * sun_mask_i / OmB        # T_sun

# Truth vector — forward model is exact so residual is purely noise
x_true   = np.concatenate([gsm_map, [T_REGOLITH, T_SUN]])
residual = np.abs(A @ x_true - y).max()
print(f"Max |A @ x_true - y| = {residual:.2f} K  (should be ~noise σ={SIGMA_NOISE} K)")

# Single thin SVD — reused for rank/condition diagnostics, inversion, and eigenmode plots
print("Computing thin SVD of A … ", end="", flush=True)
U, sv, Vt = np.linalg.svd(A, full_matrices=False)
print("done")

rank = (sv > 1e-6 * sv[0]).sum()
print(f"shape: {a.shape},  rank: {rank} / {n_unknowns}")
print(f"condition number: {sv[0]/sv[-1]:.2e}")

## Linear inversion

The `rcond` threshold zeros
out the null singular values corresponding to sky pixels that are
never in the open-sky footprint across all observations (always occluded
by the Moon or never illuminated with non-negligible beam weight).
Those pixels are flagged as unobserved in the output map.

In [ ]:
# Least-squares solution via the pre-computed thin SVD:
#   x = V @ diag(sv_inv) @ U.T @ y,  with sv_inv[k] = 1/sv[k] if sv[k] > rcond*sv[0] else 0
RCOND      = 1e-3
sv_thresh  = RCOND * sv[0]
sv_inv     = np.where(sv > sv_thresh, 1.0 / sv, 0.0)
rank_lstsq = int((sv > sv_thresh).sum())

x_est = Vt.T @ (sv_inv * (U.T @ y))

T_gsm_est = x_est[:NPIX].copy()
T_reg_est = x_est[NPIX]
T_sun_est = x_est[NPIX + 1]

# Flag pixels whose column in A is effectively zero (never in open-sky beam)
col_norms  = np.linalg.norm(A[:, :NPIX], axis=0)
unobserved = col_norms < 1e-6 * col_norms.max()
T_gsm_est[unobserved] = np.nan

print(f"Unobserved pixels: {unobserved.sum()} / {NPIX}")
obs = ~unobserved
print("Recovered scalars:")
print(f"  T_regolith : truth = {T_REGOLITH:.1f} K   est = {T_reg_est:.1f} K")
print(f"  T_sun      : truth = {T_SUN:.1f} K  est = {T_sun_est:.1f} K")
print(f"GSM monopole: truth = {np.mean(gsm_map[obs])} K  est = {np.mean(T_gsm_est[obs]} K")
print(f"GSM rms (observed pixels): {np.std(T_gsm_est[obs] - gsm_map[obs]):.1f} K")

## Singular value spectrum and sky eigenmodes

The thin SVD $A = U\,\Sigma\,V^T$ is computed once after building the design
matrix.  The right singular vectors $V[:,k]$ are the eigenmodes of $A^T A$
(eigenvalue $\sigma_k^2$); they show which sky directions the data can
constrain.  The same factorisation is used directly for the least-squares
inversion, so no matrix is inverted or decomposed more than once.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(sv / sv[0], ".-", ms=4)
ax.axhline(RCOND, color="r", ls="--", label="rcond threshold")
ax.set_xlabel("singular value index")
ax.set_ylabel("normalised singular value")
ax.set_title(f"Singular value spectrum  (rank {rank_lstsq} / {N_UNKNOWNS})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Vt[k, :NPIX] reshaped as a HEALPix map is the k-th right singular vector of A,
# i.e. the k-th eigenmode of A^T A.  sv[k]^2 is the corresponding eigenvalue.
n_show   = 6
best_idx  = list(range(n_show))
worst_idx = list(range(rank_lstsq - n_show, rank_lstsq))

fig = plt.figure(figsize=(14, 6))
for col, k in enumerate(best_idx):
    healpy.mollview(
        Vt[k, :NPIX],
        title=f"Best mode {k}  sv={sv[k]:.2e}",
        fig=fig.number, sub=(2, n_show, col + 1),
        coord="G", cmap="RdBu_r",
    )
for col, k in enumerate(worst_idx):
    healpy.mollview(
        Vt[k, :NPIX],
        title=f"Worst mode {k}  sv={sv[k]:.2e}",
        fig=fig.number, sub=(2, n_show, n_show + col + 1),
        coord="G", cmap="RdBu_r",
    )
plt.suptitle("Best (top) and worst observed (bottom) constrained sky modes", y=1.01)
plt.tight_layout()
plt.show()

## Sky coverage over 30 days

In [ ]:
## Sky coverage: sum of (B * mask / OmB) over all observations — shows which
## galactic pixels were most frequently observed with open-sky beam weight.
#coverage = np.zeros(NPIX)
#for i in range(N_OBS):
#    m = masks[i].astype(np.float64)
#    for d in range(2):
#        coverage += beams[i, d] * m / omega_B[i, d]
#
#healpy.mollview(
#    coverage,
#    title="Sky coverage (Σ beam×mask weights over all observations)",
#    unit="a.u.", cmap="viridis", coord="G",
#)
#plt.show()

## Sky map comparison

In [ ]:
T_lim   = (gsm_map.min(), gsm_map.max())
res_lim = np.nanpercentile(np.abs(T_gsm_est - gsm_map), 95)

for title, m, kw in [
    ("Truth GSM (full resolution)",  gsm_map,
     dict(min=T_lim[0], max=T_lim[1])),
    ("Recovered GSM",                T_gsm_est,
     dict(min=T_lim[0], max=T_lim[1])),
    ("Residual (recovered − truth)", T_gsm_est - gsm_map,
     dict(min=-res_lim, max=res_lim, cmap="RdBu_r")),
]:
    healpy.mollview(m, title=title, unit="K", coord="G", **kw)
    plt.show()

## Residuals and per-pixel error

In [ ]:
## Per-pixel observation sensitivity: column norm of A gives the RMS beam weight
## each sky pixel received across all observations.
#col_norm_map = np.linalg.norm(A[:, :NPIX], axis=0)
#
#healpy.mollview(
#    col_norm_map,
#    title="Per-pixel observation sensitivity  (‖A[:,j]‖)",
#    unit="a.u.", cmap="viridis", coord="G",
#)
#plt.show()

## Dipole beam orientations sampled over the mission

Plot the evolution of both dipole axes in galactic coordinates to verify
the torque-free tumbling provides diverse angular sampling.

In [ ]:
# Dipole axes in galactic frame at each observation time  (N_OBS, 2, 3)
D_gal_all = np.array([rots_obs[i].apply(U_BODY) for i in range(N_OBS)])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for d, ax in enumerate(axes):
    dx, dy, dz = D_gal_all[:, d, :].T
    sc = ax.scatter(np.arctan2(dy, dx) * 180/np.pi,
                    np.arcsin(dz)      * 180/np.pi,
                    c=t_obs_s / 86400, cmap="viridis", s=5)
    plt.colorbar(sc, ax=ax, label="day")
    ax.set_xlabel("galactic longitude [°]")
    ax.set_ylabel("galactic latitude [°]")
    ax.set_title(f"Dipole {d} axis track (galactic)")

plt.tight_layout()
plt.show()